# Лабораторная работа 4
## Оптимизация параметров модели SIR

В данном эксперименте рассматривается задача подбора параметров
агентной модели SIR. В отличие от предыдущих экспериментов, где
параметры задавались заранее и исследовалось поведение модели,
здесь требуется автоматически найти такие значения параметров,
при которых развитие эпидемии становится наиболее благоприятным
с точки зрения выбранного критерия.

В оптимизации участвуют три параметра:

- коэффициент заражения `β_und`;
- время выявления инфекции `detection_time`;
- вероятность смерти `death_rate`.

В качестве целевой функции используется суммарная оценка,
учитывающая:

- пиковую долю инфицированных;
- долю умерших.

Таким образом, задача сводится к поиску такого набора параметров,
при котором одновременно уменьшается как нагрузка на систему
здравоохранения, так и итоговые потери населения.

In [ ]:
using DrWatson
@quickactivate "project"

using BlackBoxOptim, Random, Statistics
using Agents
using JLD2

include(srcdir("sir_model.jl"))

## Целевая функция

Целевая функция принимает вектор параметров `x`, где:

- `x[1]` — коэффициент заражения `β_und`;
- `x[2]` — время выявления инфекции;
- `x[3]` — вероятность смерти.

Для уменьшения влияния случайности результат усредняется
по нескольким прогонам модели.

In [ ]:
function cost_fast(x)
infected_frac(model) =
count(a.status == :I for a in allagents(model)) / nagents(model)

dead_count(model) = 1500 - nagents(model)

peak_vals = Float64[]
dead_vals = Float64[]

for rep in 1:2
model = initialize_sir(;
Ns = [500, 500, 500],
β_und = fill(x[1], 3),
β_det = fill(x[1] / 10, 3),
infection_period = 10,
detection_time = round(Int, x[2]),
death_rate = x[3],
reinfection_probability = 0.1,
Is = [0, 0, 1],
seed = 42 + rep,
n_steps = 50,
)

peak_infected = 0.0

for step in 1:50
Agents.step!(model, 1)
frac = infected_frac(model)
if frac > peak_infected
peak_infected = frac
end
end

push!(peak_vals, peak_infected)
push!(dead_vals, dead_count(model) / 1500)
end

mean_peak = mean(peak_vals)
mean_dead = mean(dead_vals)

return mean_peak + mean_dead
end

## Запуск оптимизации

Для поиска оптимальных значений используется алгоритм случайного поиска.
Он работает быстрее более сложных многокритериальных методов и подходит
для облегчённой демонстрационной версии модели.

In [ ]:
result = bboptimize(
cost_fast;
Method = :random_search,
SearchRange = [
(0.1, 1.0),
(3.0, 14.0),
(0.01, 0.1),
],
NumDimensions = 3,
MaxTime = 20,
TraceMode = :silent,
)

## Извлечение результатов

После завершения оптимизации извлекаются найденные параметры
и значение целевой функции.

In [ ]:
best = best_candidate(result)
fitness = best_fitness(result)

println("Оптимальные параметры:")
println("β_und = $(best[1])")
println("Время выявления = $(round(Int, best[2])) дней")
println("Смертность = $(best[3])")

println("Значение целевой функции: $(fitness)")

## Сохранение результатов

Сохраним найденные параметры и значение целевой функции
для последующего использования.

In [ ]:
save(
datadir("optimization_fast_result.jld2"),
Dict(
"best" => best,
"fitness" => fitness,
),
)

## Вывод

В результате оптимизации были найдены параметры модели,
обеспечивающие минимальное значение целевой функции.
Это означает, что алгоритм подобрал такое сочетание
коэффициента заражения, времени выявления инфекции и вероятности смерти,
при котором модель демонстрирует наиболее благоприятный сценарий
в рамках заданных ограничений поиска.